Import Libraries

In [3]:
# Parallelization
import os # Operating system interface
import tempfile
import multiprocessing as mp # Package for supporting spawning processes using APi similar to threading module
from multiprocessing import Pool # Allows pooling resources into subprocesses for making effective use of multiple cpus
from multiprocessing import cpu_count # Counts the number of cpu's in current system
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor # Class for parallelization that serves an updated version of pool

# MDAnalysis
import MDAnalysis as mda # Package for general simulation analysis
from MDAnalysis.analysis import rms # MDAnalysis function for analysis of root mean square quanities
from MDAnalysis.lib.distances import distance_array

# MDTraj 
import mdtraj as md # Python library for analyzing molecular dynamics trajectories

# Itertools
from itertools import product # Product function of python's itertools module that returns cartesian product of input iterables

# Hole2
from mdahole2.analysis.hole import HoleAnalysis # Function for pore interior analysis

# PyLipid
import pylipid # Python package for analyzing lipid interactions with membrane proteins from molecular dynamics simulations
from pylipid.util import get_traj_info # Gets trajectory info regarding atom/residue index and topologies
from pylipid.util import check_dir

# Plotting
# Color blind friendly color palettes for plotting
# Sunset spectrum (blue to tan to ruby): #364B9A #4A7BB7 #6EA6CD #98CAE1 #C2E4EF #EAECCC #FEDA8B #FDB366 #F67E4B #DD3D2D #A50026
# Viridis spectrum (gold to seafoam to violet): #FDE725 #5EC962 #21918C #3B528B #440154
# Magma spectrum (yellow to magenta to black): #FCFDBF #FC8961 #B73779 #51127C #000004
# Monochromatic teal spectrum (teal to aqua to black): #FEFFFF #DEF2F1 #3AAFA9 #2B7A78 #17252A
import matplotlib.pyplot as plt # Comprehensive library for creating visualizations in Python
import networkx as nx

# NGLView
import nglview as nv # Package for interactive visualization of proteins and trajectories

# NumPY
import numpy as np # Fundamental package for mathematical operations and numerical computations in Python

# SciPy
import scipy # Library that provides fundamental algorithms for optimization, integration, interpolation, and statistics
from scipy.ndimage import gaussian_filter1d # SciPy libtrary for gaussian filter (plot/image smoothing)
from scipy.spatial import ConvexHull # SciPy library that can take an index of points and find whether they form a convex hull
from scipy.spatial import Delaunay # SciPy library that supports Delaunay triangulation
from scipy.spatial import cKDTree # Scipy class that provides an index into k-dimensional points that finds nearest neighbors
from scipy.interpolate import interp1d # Scipy library for piecewise continuous interpolation
from scipy.interpolate import make_interp_spline
from scipy.signal import savgol_filter

# Pandas
import pandas as pd # Python library for data structure and data analysis

# Warnings
import warnings # Python core function for handling built-in exceptions
warnings.filterwarnings('ignore') # Ignore unnessary warnings from MDAnalysis

Analysis and Computing Class for Lipids and Lipid Interactions

In [ ]:
# Class for everything involving simulation lipid properties.
class LipidAnalysis:
    StructureFileTypes = {'.gro'} # Available structure file types
    TrajectoryFileTypes = {'.xtc'} # Available trajectory file types

    # Function for initiating protein universes
    def __init__(self,Structure,Trajectory,FrameStep=100,BatchSize=10):
        if all(isinstance(Struct,str) for Struct in Structure): # If structure input is a string, assigns to structure
            Structures = Structure
        elif not isinstance(Structure,list): # Otherwise outputs an error for wrong input type
            raise TypeError("Structure must either be a string or a list of strings.")
        if all(isinstance(Traj,str) for Traj in Trajectory): # If trajectory exists and is a string/list of strings, creates trajectory object
            Trajectories = Trajectory
        elif not isinstance(Trajectory,list): # If provided and wrong file type, outputs error
            raise TypeError("Trajectory must be a string or a list of strings.")
        if len(Structures) != len(Trajectories): # If there are not the same amount of structures and trajectories, produce error
            raise ValueError("The number of structures and trajectories must be the same.")
        for Struct in Structures: # Checks for every structure if in supported structure file types
            if os.path.splitext(Struct)[1].lower() not in self.StructureFileTypes:
                raise ValueError(f"File type {os.path.splitext(Struct)[1].lower()} is unsupported for structures.")
        for Traj in Trajectories: # Checks for every trajectory if in supported file types
            if Traj is not None:
                if os.path.splitext(Traj)[1].lower() not in self.TrajectoryFileTypes:
                    raise ValueError(f"File type {os.path.splitext(Traj)[1].lower()} is unsupported for trajectories.")
        # Creates universe from structures and optionally and trajectory
        self.Universe = [mda.Universe(Struct,Traj) for Struct,Traj in zip(Structures,Trajectories)]
        self.StructureList = Structures
        self.TrajectoryList = Trajectories
        self.FrameStep = FrameStep # Saves frame step, i.e. with what frequency are frames read
        self.BatchSize = BatchSize # Batchsize for parallel processing

    def LipidDistance(self,Lipid,LipidAtoms=None,ProteinAtomSelection="all",StartFrame=0,EndFrame=None):
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
            Futures = []
            for Structure, TrajectoryFile in zip(self.StructureList, self.TrajectoryList):
                Futures.append(executor.submit(LipidAnalysis.LipidDistanceCalculation,Structure,TrajectoryFile,
                        Lipid,LipidAtoms,ProteinAtomSelection,self.FrameStep,StartFrame,EndFrame))
            Results = [Future.result() for Future in Futures]
        MaxContacts = max(len(R) for R in Results)
        NumTrajectories = len(Results)
        NumFrames = Results[0][0].shape[0] if Results[0] else 0
        DistanceContactsArray = np.full((NumTrajectories, MaxContacts, NumFrames), np.nan)
        for TrajIndex, Contacts in enumerate(Results):
            for ContactIndex, ContactArray in enumerate(Contacts):
                DistanceContactsArray[TrajIndex, ContactIndex, :] = ContactArray
        self.DistanceContacts = DistanceContactsArray

    @staticmethod
    def LipidDistanceCalculation(Structure,TrajectoryFile,Lipid,LipidAtoms,ProteinAtomSelection,FrameStep,StartFrame,EndFrame):
        Traj = md.load(TrajectoryFile, top=Structure)
        if EndFrame is None:
            EndFrame = Traj.n_frames
        Traj = Traj[StartFrame:EndFrame:FrameStep]
        TrajInfo, _, _ = get_traj_info(Traj, Lipid, lipid_atoms=LipidAtoms)
        DistanceContacts = []
        topology = Traj.topology
        all_protein_atoms = [atom for atom in topology.atoms if atom.residue.is_protein]
        if ProteinAtomSelection != "all":
            if isinstance(ProteinAtomSelection, str):
                selected_atoms = [a for a in all_protein_atoms if a.name == ProteinAtomSelection]
            elif isinstance(ProteinAtomSelection, (list, tuple, set)):
                selected_atoms = [a for a in all_protein_atoms if a.name in ProteinAtomSelection]
            else:
                raise ValueError("ProteinAtomSelection must be 'all', a string, or a list of atom names.")
            selected_indices = [a.index for a in selected_atoms]
        else:
            selected_indices = [a.index for a in all_protein_atoms]
        for ResidueIndex, ResidueAtomIndex in enumerate(TrajInfo["protein_residue_atomid_list"]):
            ResidueAtomIndex = [int(Atom) for Sublist in ResidueAtomIndex for Atom in (Sublist if isinstance(Sublist, list) else [Sublist])]
            ResidueAtomIndex = [i for i in ResidueAtomIndex if i in selected_indices]
            if not ResidueAtomIndex:
                continue
            DistanceList = []
            for LipidAtomIndex in TrajInfo["lipid_residue_atomid_list"]:
                LipidAtomIndex = [int(Atom) for Sublist in LipidAtomIndex for Atom in (Sublist if isinstance(Sublist, list) else [Sublist])]
                AtomPairs = np.array([[i, j] for i in ResidueAtomIndex for j in LipidAtomIndex], dtype=int)
                if AtomPairs.shape[0] == 0:
                    continue
                Distances = md.compute_distances(Traj, AtomPairs)
                DistanceList.append(np.min(Distances, axis=1))
            if len(DistanceList) == 0:
                DistanceMatrix = np.empty((0, Traj.n_frames))
            else:
                DistanceMatrix = np.stack(DistanceList, axis=0)
            for LipidIndex in range(len(DistanceMatrix)):
                DistanceContacts.append(DistanceMatrix[LipidIndex])
        return DistanceContacts

    '''# Method for finding minimum distance array between lipids and protein over trajectories
    def LipidDistance(self,Lipid,LipidAtoms=None,StartFrame=0,EndFrame=None):
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor: # Initiates pooling of resources for parallelization
            Futures = [] # Stores outputs for addendum to results
            # Loops over universes and finds the minimum distances contacts for each universe using parallelized static method
            for Structure, TrajectoryFile in zip(self.StructureList, self.TrajectoryList):
                Futures.append(executor.submit(IonChannelLA.LipidDistanceCalculation,Structure,TrajectoryFile,Lipid,LipidAtoms,
                                               self.FrameStep,StartFrame,EndFrame))
            Results = [Future.result() for Future in Futures] # Summarizes results from calculations into one array
        MaxContacts = max(len(R) for R in Results) # Calculates the maximum number of contacts 
        NumTrajectories = len(Results) # Finds the number of trajectories analysis was done over
        NumFrames = Results[0][0].shape[0] if Results[0] else 0 # Finds number of processed frames per trajectory 
        DistanceContactsArray = np.full((NumTrajectories,MaxContacts,NumFrames),np.nan) # Creates an array of distance contacts
        for TrajIndex, Contacts in enumerate(Results): # Loops over trajectories in results
            for ContactIndex, ContactArray in enumerate(Contacts): # Loopds over contacts in results
                DistanceContactsArray[TrajIndex,ContactIndex,:] = ContactArray # Fills distance contact array from results
        self.DistanceContacts = DistanceContactsArray # Outputs distance contact array between protein and lipids

    # Method for calculating actual minima between lipids and proteins
    @staticmethod # Static method for pooling
    def LipidDistanceCalculation(Structure,TrajectoryFile,Lipid,LipidAtoms,FrameStep,StartFrame,EndFrame):
        Traj = md.load(TrajectoryFile, top=Structure) # Loads trajectory in md
        if EndFrame == None: # If the endframe is empty, stes endframe as last frame in trajectory
            EndFrame = Traj.n_frames
        Traj = Traj[StartFrame:EndFrame:FrameStep] # Stes the frames for study in the trajectory, skipping to every nth frame
        TrajInfo, _, _ = get_traj_info(Traj,Lipid,lipid_atoms=LipidAtoms) # Function to find lipid trajectory info
        DistanceContacts = [] # Array for storing distance contacts
        # Loop for each residue and atom in the protein
        for ResidueIndex, ResidueAtomIndex in enumerate(TrajInfo["protein_residue_atomid_list"]):
            DistanceList = [] # List for storing distances from the protein atoms to lipids
            # Creates atom index for protein
            ResidueAtomIndex = [int(Atom) for Sublist in ResidueAtomIndex for Atom in 
                                (Sublist if isinstance(Sublist, list) else [Sublist])]
            # For each lipid, creates a lipid atom index
            for LipidAtomIndex in TrajInfo["lipid_residue_atomid_list"]:
                LipidAtomIndex = [int(Atom) for Sublist in LipidAtomIndex for Atom in 
                                  (Sublist if isinstance(Sublist, list) else [Sublist])]
                # Pairs lipid atoms and residue atoms to find minimum distances
                AtomPairs = np.array([[i,j] for i in ResidueAtomIndex for j in LipidAtomIndex], dtype=int)
                if AtomPairs.shape[0] == 0: # If no atompairs found (error), continues loop
                    continue
                Distances = md.compute_distances(Traj,AtomPairs) # Finds distance between atom pairs
                DistanceList.append(np.min(Distances, axis=1)) # Appends distances found to list of all minimum distances
            if len(DistanceList) == 0: 
                DistanceMatrix = np.empty((0,Traj.n_frames)) # For an empty distance list, outputs no matchings
            else: 
                DistanceMatrix = np.stack(DistanceList, axis=0) # Adds up lists of distances for lipids into a matrix
            for LipidIndex in range(len(DistanceMatrix)):
                DistanceContacts.append(DistanceMatrix[LipidIndex]) # Appends the distance matrix per lipid to the contacts
        return DistanceContacts  # Returns matrix of lipid distance contacts wioth protein for one trajectory'''

CHOL Binding Calculations

In [5]:
LipidCHOL = "CHOL"
LipidAtoms = "ROH" # Choleterol head group
ProteinAtom = "BB"

Structure = ["/lustre/hdd/LAS/potoyan-lab/zach/TRPV1Simulations/CoarseCompletedV2/325K15/EquilibrationStep4.gro"]
Trajectory = ["/lustre/hdd/LAS/potoyan-lab/zach/TRPV1Simulations/CoarseCompletedV2/325K15/NPT.xtc"]

LA325K15CHOL = LipidAnalysis(Structure,Trajectory)
LA325K15CHOL.LipidDistance(LipidCHOL,LipidAtoms=LipidAtoms,ProteinAtomSelection=ProteinAtom)

In [10]:
CHOLContacts325K15 = np.array(LA325K15CHOL.DistanceContacts)
print(CHOLContacts325K15[0,2,20])

1.5705420970916748


POPC Binding Calculations